# Traces → SFT Distillation

Distil a deployed Foundry hosted agent's tool-using behavior into a smaller, cheaper student model — using **real production traces** as training data. No labeling required.

End-to-end:
1. Pull conversation traces from a deployed Foundry agent via the Data Generation API
2. Transform the raw export into Azure-FT-ready JSONL (6 fixes the trace export currently needs)
3. Split into train / val / test
4. Score the base student on a held-out test set with **structural tool-call matching**
5. Submit one fine-tuning job (winning hyperparameters)
6. Monitor training to completion
7. Deploy the fine-tuned model
8. Score it on the same test set and report the lift

**Reference result on a deployed Zava retail agent** (~100 conversations, `gpt-4.1-nano` student):
- Base `gpt-4.1-nano`: 7.38/10, 60% pass rate
- Fine-tuned `gpt-4.1-nano` (3 epochs, lr=1.0): **8.60/10, +16.5% lift, 60%→100% pass**

> ⚠️ `gpt-4.1-nano` (2025-04-14) is now in **`Deprecating`** lifecycle state and is rejected for
> *new* deployments (`ServiceModelDeprecating`). `STUDENT_MODEL` below therefore defaults to
> **`gpt-4.1-mini`**. Expect a smaller lift than the numbers above: mini starts closer to a
> `gpt-4.1` teacher than nano did, so there is less headroom to recover. The numbers above are
> kept as the published reference point, not as the result you should expect from this run.

**Cost**: ~$3–8 per run. **Time**: ~30–50 minutes.

This notebook is **fully self-contained** — no external skill scripts required. All helpers (Foundry traces datagen, 6-step transform, tool-call evaluator) are defined inline.

## 1. Setup & Configuration

In [ ]:
import os, json, time, random, re, subprocess, sys
from pathlib import Path
from openai import OpenAI
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]   # project where the hosted agent + its traces live
BASE_URL         = os.environ["OPENAI_BASE_URL"]              # https://<resource>.openai.azure.com/openai/v1

# The fine-tuning resource has key auth disabled (disableLocalAuth=true), so mint an
# AAD token instead of reading AZURE_OPENAI_API_KEY. Tokens last ~60-90 min and a
# training run can outlive one, so long-running loops call new_client() to refresh.
_cred = DefaultAzureCredential()

def new_client():
    tok = _cred.get_token("https://cognitiveservices.azure.com/.default").token
    return OpenAI(base_url=BASE_URL, api_key=tok)

AGENT_NAME       = "zava-trace-teacher"         # deployed agent on PROJECT_ENDPOINT
AGENT_VERSION    = "1"
HOURS_LOOKBACK   = 720                          # 30 days

STUDENT_MODEL    = "gpt-4.1-mini"               # the model we are distilling INTO

WORK             = Path("./run").resolve()
WORK.mkdir(exist_ok=True)

client = new_client()
print(f"Project:   {PROJECT_ENDPOINT}")
print(f"Agent:     {AGENT_NAME}:{AGENT_VERSION}")
print(f"Lookback:  {HOURS_LOOKBACK} hours")
print(f"Student:   {STUDENT_MODEL}")


## 2. Inline helper: Foundry traces datagen via SDK

In [ ]:
def submit_foundry_traces_datagen(*, agent_name, agent_version, hours,
                                  max_samples, output_name, project_endpoint, timeout_s=1800):
    """Submit a Foundry Traces datagen job that pulls historical conversations from App Insights.

    Returns the path to the downloaded raw JSONL.
    """
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import (
        DataGenerationJob, DataGenerationJobInputs,
        DataGenerationJobScenario, DataGenerationJobOutputOptions,
        TracesDataGenerationJobOptions, TracesDataGenerationJobSource,
    )
    from azure.identity import DefaultAzureCredential
    from datetime import datetime, timedelta, timezone

    project = AIProjectClient(endpoint=project_endpoint, credential=DefaultAzureCredential())

    # The traces source takes an explicit time window, not a lookback duration
    # (`hours=` was removed from TracesDataGenerationJobSource), and the job inputs
    # now require a `name`.
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours)

    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[TracesDataGenerationJobSource(
            agent_name=agent_name,
            agent_version=agent_version,
            start_time=start_time,
            end_time=end_time,
        )],
        options=TracesDataGenerationJobOptions(max_samples=max_samples),
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    print(f"Submitting traces datagen job (agent={agent_name}:{agent_version}, hours={hours}, max={max_samples})...")
    # Datagen moved to the beta operation group and now returns an LRO poller
    # instead of a job object you poll by hand.
    poller = project.beta.datasets.begin_create_generation_job(job)
    print(f"  job.id={poller.details.get('id')}  status={poller.status()}")

    poller.wait(timeout=timeout_s)
    if not poller.done():
        raise TimeoutError(f"traces datagen job did not complete in {timeout_s}s")
    result = poller.result()

    # Each output carries a single Azure OpenAI file id (previously a file_ids list).
    file_ids = [o.id for o in (result.outputs or []) if getattr(o, "id", None)]
    print(f"  generated {result.generated_samples} sample(s) across {len(file_ids)} file(s)")
    if not file_ids:
        raise RuntimeError("traces datagen job produced no output files")

    # The output files live in the project that ran the job, which is not
    # necessarily the resource `client` points at (here: agent project in eastus2
    # vs. fine-tuning resource in northcentralus). Download via the project's own
    # OpenAI client so the file IDs resolve.
    project_oai = project.get_openai_client()
    out_path = WORK / f"{output_name}_dg.jsonl"
    out_blob = b""
    for fid in file_ids:
        out_blob += project_oai.files.content(file_id=fid).read()
    out_path.write_bytes(out_blob)
    print(f"  saved {out_path.name}  ({out_path.stat().st_size:,} bytes)")
    return out_path


## 3. Inline helpers: 5-step transform for Foundry traces export

Foundry's traces export currently emits JSONL with five issues that Azure FT preprocessing rejects:
1. **Overlapping snapshots** — each LangGraph node invocation produces a span; the worker stitches them into one row, so one row contains N overlapping snapshots
2. **Fragments** — when the customer simulator ends right after an asst clarifying question, you get 2-msg rows with no tool_calls (no SFT signal)
3. **content="null"** — assistant tool-call rows have content as the literal string `"null"` (Azure FT requires content omitted entirely on tool-call rows)
4. **Consecutive asst tool_call turns** — sometimes two adjacent asst spans each issue one call (Azure FT requires tool replies immediately after each asst turn)
5. **Missing system + tools** — the trace export does not currently emit system or tools at row level; tool-using FT needs both

These five functions apply all five fixes. As Foundry's export improves these should become no-ops, but they're idempotent so the notebook is safe to keep.

In [ ]:
def dedup_messages(msgs):
    """Collapse overlapping snapshots via first-occurrence dedup."""
    def key(m):
        tcs = m.get("tool_calls") or []
        tc_key = tuple(
            (tc.get("id"),
             (tc.get("function") or {}).get("name"),
             (tc.get("function") or {}).get("arguments"))
            for tc in tcs
        )
        return (m.get("role"), m.get("content") or "", m.get("tool_call_id"), tc_key)
    seen, out = set(), []
    for m in msgs:
        k = key(m)
        if k in seen: continue
        seen.add(k); out.append(m)
    return out, len(msgs) - len(out)


def is_fragment(msgs):
    """Row is a fragment if it has no assistant tool_calls (nothing to learn)."""
    return not any(m.get("role") == "assistant" and m.get("tool_calls") for m in msgs)


def merge_consecutive_asst_tool_calls(msgs):
    """Merge runs of consecutive assistant messages with tool_calls into one
    assistant message with the tool_calls array combined."""
    out, merged, i = [], 0, 0
    while i < len(msgs):
        m = msgs[i]
        if m.get("role") == "assistant" and m.get("tool_calls"):
            combined = dict(m); combined["tool_calls"] = list(m["tool_calls"])
            j = i + 1
            while j < len(msgs) and msgs[j].get("role") == "assistant" and msgs[j].get("tool_calls"):
                combined["tool_calls"].extend(msgs[j]["tool_calls"]); merged += 1; j += 1
            out.append(combined); i = j
        else:
            out.append(m); i += 1
    if merged: msgs[:] = out
    return merged


def fix_null_content(msgs):
    """Remove the content key entirely on assistant tool-call rows. Azure FT
    preprocessing rejects rows where content is present alongside tool_calls."""
    n = 0
    for m in msgs:
        if m.get("role") == "assistant" and m.get("tool_calls") and "content" in m:
            m.pop("content"); n += 1
    return n


def fix_tool_call_ids(msgs, row_idx=0):
    """Re-link tool replies to the assistant turn that requested them.

    The traces export emits every tool message with an empty ``tool_call_id`` while
    the assistant turn carries the real ``call_...`` id. Azure FT preprocessing
    cannot pair them and rejects the row with "contains invalid schema" — only rows
    that actually contain tool replies are affected. Assistant tool_calls sometimes
    carry an empty id too, so synthesise one first, then hand the ids out to the
    following tool messages in order.
    """
    n = 0
    pending = []
    for m in msgs:
        role = m.get("role")
        if role == "assistant" and m.get("tool_calls"):
            pending = []
            for i, tc in enumerate(m["tool_calls"]):
                if not tc.get("id"):
                    tc["id"] = f"call_r{row_idx}_{i}_{tc.get('function', {}).get('name', 'fn')}"[:40]
                    n += 1
                pending.append(tc["id"])
        elif role == "tool":
            if pending:
                want = pending.pop(0)
                if m.get("tool_call_id") != want:
                    m["tool_call_id"] = want; n += 1
    return n


def transform_traces_for_ft(raw_path, system_prompt, tools, out_path):
    """Apply the six fixes Azure FT preprocessing requires on the Foundry traces export."""
    rows_in, rows_out = 0, 0
    with open(raw_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            rows_in += 1
            row = json.loads(line)
            msgs = row.get("messages") or []
            msgs, _ = dedup_messages(msgs)
            if is_fragment(msgs): continue
            merge_consecutive_asst_tool_calls(msgs)
            fix_null_content(msgs)
            fix_tool_call_ids(msgs, rows_in)
            # The traces export now emits the system message itself. Prepending
            # unconditionally produced two consecutive system turns, which Azure FT
            # rejects at preprocessing with "contains invalid schema". Drop any
            # exported system turns and keep exactly one, so this stays idempotent
            # whether or not the export includes one.
            body = [m for m in msgs if m.get("role") != "system"]
            row["messages"] = [{"role": "system", "content": system_prompt}] + body
            row["tools"] = tools
            row["parallel_tool_calls"] = True
            fout.write(json.dumps(row) + "\n")
            rows_out += 1
    print(f"  Transformed {rows_in} raw -> {rows_out} clean rows")
    return out_path


## 4. Pull traces and transform into FT-ready JSONL

In [ ]:
# Pull historical traces for your agent (last 720 hours)
RAW_DATA = submit_foundry_traces_datagen(
    agent_name=AGENT_NAME,
    agent_version=AGENT_VERSION,
    hours=HOURS_LOOKBACK,
    max_samples=100,
    output_name="traces-raw",
    project_endpoint=PROJECT_ENDPOINT,
)

# Apply the 5-step transform
SYSTEM_PROMPT = Path("fixtures/zava_system_prompt.md").read_text(encoding="utf-8")
TOOLS         = json.loads(Path("fixtures/zava_tools.json").read_text())

CLEAN_DATA = WORK / "traces-clean.jsonl"
transform_traces_for_ft(RAW_DATA, SYSTEM_PROMPT, TOOLS, CLEAN_DATA)

with open(CLEAN_DATA, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]
print(f"\nCleaned: {len(data)} rows")


## 5. Split into train / val / test

In [ ]:
rng = random.Random(42)
indices = list(range(len(data))); rng.shuffle(indices)
n_train = int(0.8 * len(indices)); n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]; val_idx = indices[n_train:n_train+n_val]; test_idx = indices[n_train+n_val:]

TRAIN_PATH = WORK / "train.jsonl"; VAL_PATH = WORK / "val.jsonl"; TEST_PATH = WORK / "test.jsonl"
for path, idxs in [(TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)]:
    with path.open("w", encoding="utf-8") as f:
        for i in idxs: f.write(json.dumps(data[i]) + "\n")

print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


## 6. Baseline evaluation via the Foundry evals SDK

We use `azure-ai-evaluation.evaluate()` as the driver. The built-in evaluators don't cover tool-call structural matching, so we provide a small custom evaluator function the SDK runs across all test rows.

In [ ]:
def tool_call_score(ref_calls, out_calls):
    if not ref_calls: return 10 if not out_calls else 5
    if not out_calls: return 1
    def _name(c):
        if isinstance(c, dict): return ((c.get("function") or {}).get("name")) or c.get("name")
        fn = getattr(c, "function", None)
        return getattr(fn, "name", None) if fn else getattr(c, "name", None)
    def _args(c):
        if isinstance(c, dict):
            raw = ((c.get("function") or {}).get("arguments")) or c.get("arguments")
        else:
            fn = getattr(c, "function", None)
            raw = getattr(fn, "arguments", None) if fn else getattr(c, "arguments", None)
        if isinstance(raw, str):
            try: return json.loads(raw)
            except Exception: return {"_raw": raw}
        return raw or {}
    ref_names = [_name(c) for c in ref_calls]
    out_names = [_name(c) for c in out_calls]
    ref_set, out_set = set(ref_names), set(out_names)
    overlap = ref_set & out_set
    if not overlap: return 1
    if ref_set != out_set:
        ratio = len(overlap) / len(ref_set | out_set)
        return max(2, int(round(2 + ratio * 6)))
    ref_args = {_name(c): _args(c) for c in ref_calls}
    out_args = {_name(c): _args(c) for c in out_calls}
    return 10 if all(ref_args[n] == out_args.get(n) for n in ref_names) else 8


def tool_call_evaluator(*, response, ground_truth, **kwargs):
    ref = json.loads(ground_truth) if isinstance(ground_truth, str) else (ground_truth or [])
    out = json.loads(response) if isinstance(response, str) else (response or [])
    score = tool_call_score(ref, out)
    return {"tool_match_score": score, "tool_match_pass": 1.0 if score >= 8 else 0.0}


def make_target(model_name):
    def _target(*, prompt, system, **kwargs):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role":"system","content":system}, {"role":"user","content":prompt}],
                tools=TOOLS, temperature=0.0, max_completion_tokens=2048,
            )
            tool_calls = resp.choices[0].message.tool_calls or []
            return {"response": json.dumps([
                {"function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in tool_calls
            ])}
        except Exception as e:
            return {"response": "[]", "error": str(e)[:200]}
    return _target


# Build eval data from the test set
EVAL_DATA_PATH = WORK / "eval_data.jsonl"
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip(): continue
        row = json.loads(line)
        msgs = row["messages"]
        user_msg = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
        first_asst = next((m for m in msgs if m.get("role") == "assistant"), {})
        ref_tcs = first_asst.get("tool_calls") or []
        gt = [{"function": {"name": (tc.get("function") or {}).get("name"),
                            "arguments": (tc.get("function") or {}).get("arguments")}} for tc in ref_tcs]
        fout.write(json.dumps({"system": SYSTEM_PROMPT, "prompt": user_msg, "ground_truth": json.dumps(gt)}) + "\n")
print(f"Eval data: {EVAL_DATA_PATH.name}")

from azure.ai.evaluation import evaluate as _evaluate


def evaluate(**kwargs):
    """Run the evaluation SDK without losing the notebook's output stream.

    When called with `target=`, promptflow's batch executor swaps out the
    kernel's stdout/stderr and never puts them back, so every cell after the
    first evaluation captured nothing -- including the final results table.
    Snapshot both around the call and restore them.
    """
    saved = sys.stdout, sys.stderr
    try:
        return _evaluate(**kwargs)
    finally:
        sys.stdout, sys.stderr = saved

print(f"\nBaseline ({STUDENT_MODEL}) evaluation...")
baseline_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(STUDENT_MODEL),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "baseline_eval_results.json"),
)
baseline_combined = baseline_results["metrics"].get("tool_match.tool_match_score", 0)
baseline_pass = baseline_results["metrics"].get("tool_match.tool_match_pass", 0) * 100
print(f"  Baseline: combined={baseline_combined:.2f}/10  pass_rate={baseline_pass:.1f}%")


## 7. Submit the fine-tuning job

Winning hyperparameters from prior experiments: **3 epochs, learning-rate multiplier 1.0** (tuned on a `gpt-4.1-nano` student; they transfer fine to `gpt-4.1-mini`). The job trains whatever `STUDENT_MODEL` is set to in section 1.

In [ ]:
print("Uploading train + val files...")
with open(TRAIN_PATH, "rb") as fh:
    train_file = client.files.create(file=(TRAIN_PATH.name, fh), purpose="fine-tune")
with open(VAL_PATH, "rb") as fh:
    val_file = client.files.create(file=(VAL_PATH.name, fh), purpose="fine-tune")

for f in (train_file, val_file):
    for _ in range(30):
        f = client.files.retrieve(file_id=f.id)
        if f.status == "processed": break
        time.sleep(2)
    print(f"  {f.id} status={f.status}")

print("\nSubmitting fine-tuning job...")
ft_job = client.fine_tuning.jobs.create(
    model=STUDENT_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
    method={"type": "supervised"},
    hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
    suffix="traces-distil-demo",
    extra_body={"trainingType": "globalStandard"},
)
print(f"  Job: {ft_job.id}  status={ft_job.status}")


## 8. Monitor training

In [ ]:
print(f"Monitoring job {ft_job.id}...\n")
last_seen_step = -1
while True:
    client = new_client()   # refresh the AAD token; training can outlast a single one
    job = client.fine_tuning.jobs.retrieve(ft_job.id)
    events = list(client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
    for e in reversed(events):
        msg = e.message or ""
        if "Step " in msg and ":" in msg:
            try:
                step = int(msg.split("Step ")[1].split(":")[0])
                if step > last_seen_step:
                    print(f"  {time.strftime('%H:%M:%S')}  {msg[:90]}")
                    last_seen_step = step
            except Exception: pass
    if job.status in ("succeeded", "failed", "cancelled"):
        print(f"\nFinal status: {job.status}")
        if job.status != "succeeded":
            print(f"Error: {job.error.message if job.error else '(none)'}")
            raise RuntimeError(f"Fine-tuning {job.status}")
        FT_MODEL_ID = job.fine_tuned_model
        print(f"Fine-tuned model: {FT_MODEL_ID}")
        break
    time.sleep(30)


## 9. Deploy the fine-tuned model

In [ ]:
m = re.match(r"https://([^.]+)\.openai\.azure\.com", BASE_URL)
ACCOUNT_NAME    = m.group(1)
SUBSCRIPTION_ID = os.environ.get("AZURE_SUBSCRIPTION_ID") or input("Azure subscription id: ")
RESOURCE_GROUP  = os.environ.get("AZURE_RESOURCE_GROUP") or input("Azure resource group: ")

DEPLOY_NAME = "traces-distil-demo"
deploy_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/deployments/{DEPLOY_NAME}?api-version=2024-10-01"
)
body = {"sku": {"name": "GlobalStandard", "capacity": 100},
        "properties": {"model": {"format": "OpenAI", "name": FT_MODEL_ID, "version": "1"}}}

import urllib.request, urllib.error
token = subprocess.check_output(["az", "account", "get-access-token", "--query", "accessToken", "-o", "tsv"]).decode().strip()
req = urllib.request.Request(deploy_url, method="PUT",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    data=json.dumps(body).encode())
try:
    urllib.request.urlopen(req)
    print(f"Deployment submitted: {DEPLOY_NAME}")
except urllib.error.HTTPError as e:
    # Fail loudly. Printing and continuing meant a rejected PUT (quota, bad
    # model id, wrong resource) spun the readiness loop for 10 minutes against
    # a deployment that never existed, then surfaced two cells later as an
    # unrelated-looking evaluation error.
    raise RuntimeError(f"Deployment failed: {e.code} {e.reason}\n{e.read().decode()[:600]}") from None

print("Waiting for deployment to become inferenceable (~3-5 min)...")
last_err = None
for i in range(20):
    try:
        client = new_client()
        client.chat.completions.create(model=DEPLOY_NAME, messages=[{"role":"user","content":"hi"}], max_completion_tokens=5)
        print(f"\n  Ready after {i*30}s")
        break
    except Exception as e:
        last_err = e
        time.sleep(30); print(".", end="", flush=True)
else:
    raise RuntimeError(f"{DEPLOY_NAME} still not inferenceable after 10 min. Last error: {last_err}")


## 10. Evaluate the fine-tuned model and compare

In [ ]:
print(f"Fine-tuned ({STUDENT_MODEL}) evaluation...")
ft_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(DEPLOY_NAME),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "ft_eval_results.json"),
)
ft_combined = ft_results["metrics"].get("tool_match.tool_match_score", 0)
ft_pass = ft_results["metrics"].get("tool_match.tool_match_pass", 0) * 100

print()
print("-" * 60)
print(f"  {'Model':<35}  {'Combined':>10}  {'Pass Rate':>10}")
print(f"  {'-'*35}  {'-'*10}  {'-'*10}")
print(f"  Baseline ({STUDENT_MODEL}){' '*(35-19-len(STUDENT_MODEL))}  {baseline_combined:>10.2f}  {baseline_pass:>9.1f}%")
print(f"  Fine-tuned ({STUDENT_MODEL}){' '*(35-21-len(STUDENT_MODEL))}  {ft_combined:>10.2f}  {ft_pass:>9.1f}%")

lift = (ft_combined - baseline_combined) / baseline_combined * 100 if baseline_combined > 0 else 0
print(f"\n  Lift:  {lift:+.1f}%")
if lift >= 5:
    print(f"  Distillation improved combined score by {lift:+.1f}% -- the small student matches the teacher agent at much lower inference cost.")
else:
    print(f"  Lift below 5% threshold. Consider: more trace data, different HPs, or a different student model.")


## (Optional) Populate trace history first

If your agent has no trace history yet, the `fixtures/push_prompts.py` script sends ~500 diverse retail-style prompts through your agent (traces land in App Insights within ~90 seconds). Run it before cell 4 (`Pull traces and transform...`):

```bash
python fixtures/push_prompts.py \
    --agent-name <your-hosted-agent> \
    --agent-version <version> \
    --num-prompts 500 \
    --concurrency 4 \
    --project-endpoint $AZURE_AI_PROJECT_ENDPOINT
```

## Cleanup

```python
req = urllib.request.Request(deploy_url, method="DELETE", headers={"Authorization": f"Bearer {token}"})
urllib.request.urlopen(req)
client.files.delete(train_file.id)
client.files.delete(val_file.id)
```

## Bring your own agent

Replace `AGENT_NAME` / `AGENT_VERSION` and the two fixture files (`zava_system_prompt.md`, `zava_tools.json`) with your agent's system prompt + tool catalog (OpenAI chat-completions format).

## Dependencies

```
pip install openai>=2.0 azure-ai-projects>=2.2.0 azure-identity>=1.21 azure-ai-evaluation>=1.0
```